# GX11 — Autovalores em forma fechada e verificação da função de Green

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/gx11_autovalores.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*  
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## Descrição

O problema GX11 corresponde à placa plana com **temperatura prescrita** em ambas as faces ($x=0$ e $x=L$, tipo 1). É o caso *mais simples* da família tratada no Cap. 4 e cumpre, aqui, o papel pedagógico de referência:

$$\sin(\lambda_n L) = 0 \quad\Longrightarrow\quad \lambda_n = \frac{n\pi}{L},\qquad n = 1, 2, 3, \ldots$$

Diferentemente dos problemas GX13 e GX23 (raízes transcendentais — ver `gx13_autovalores.ipynb` e `gx23_autovalores.ipynb`), os autovalores **não exigem método numérico** (Brent ou Newton-Raphson): a expressão é fechada e atinge precisão de máquina por construção.

Este *notebook* tem três objetivos:

1. apresentar os autovalores adimensionais $\xi_n = \beta_n L = n\pi$;
2. visualizar as autofunções $\sin(\beta_n x/L)$ e verificar **numericamente** a ortogonalidade;
3. avaliar a convergência da série da função de Green $G_{X11}$ em função do número de modos e do tempo adimensional $\alpha (t-\tau)/L^2$.

## Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize': (7.0, 4.2),
    'font.size':       11,
    'axes.grid':       True,
    'grid.alpha':      0.3,
})

## Parâmetros

Para o problema X11 os autovalores não dependem de nenhum parâmetro físico (não há $B = hL/k$ como nos casos convectivos): dependem apenas da geometria $L$ e do índice $n$. Aqui usamos $L = 1$ sem perda de generalidade — todas as quantidades aparecem em forma adimensional ($\xi_n = \beta_n L$, $x/L$, $\alpha (t-\tau)/L^2$).

In [ ]:
L = 1.0      # comprimento da placa (adimensional)
M = 5        # número de autovalores mostrados na tabela

## Autovalores em forma fechada

$$\boxed{\,\lambda_n = \frac{n\pi}{L}\,},\qquad \xi_n = \beta_n L = n\pi,\qquad n = 1, 2, 3, \ldots$$

Observe que $n = 0$ produz autofunção identicamente nula e os índices negativos repetem as autofunções a menos do sinal — por isso não são contados.

In [ ]:
n   = np.arange(1, M+1)
xi  = n * np.pi              # ξ_n = β_n · L
lam = xi / L                  # λ_n = n π / L

df = pd.DataFrame({
    'n'                : n,
    'ξ_n = β_n·L'      : xi,
    'λ_n = n π / L'    : lam,
    'sin(ξ_n)'         : np.sin(xi),
})
df = df.set_index('n')

pd.set_option('display.float_format', '{:.10g}'.format)
print(df.to_string())

A coluna `sin(ξ_n)` exibe valores da ordem de $10^{-16}$, o ruído de arredondamento esperado em ponto flutuante de dupla precisão. Confirma que os $\xi_n = n\pi$ satisfazem a equação $\sin(\xi_n) = 0$ na precisão da máquina, sem necessidade de refinamento iterativo.

## Autofunções $\sin(\beta_n x / L)$

As cinco primeiras autofunções no domínio $0 \le x/L \le 1$:

In [ ]:
x = np.linspace(0, L, 401)

fig, ax = plt.subplots()
estilos = ['-', '--', '-.', ':', (0, (3, 1, 1, 1))]
for k, beta in enumerate(xi / L):
    ax.plot(x/L, np.sin(beta*x), estilos[k], lw=1.8,
            label=fr'$n={k+1},\ \beta_n L = {(k+1)}\pi$')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel(r'$x/L$')
ax.set_ylabel(r'$\sin(\beta_n x / L)$')
ax.set_title('Autofunções do problema GX11')
ax.legend(loc='lower center', ncol=3, fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()

## Verificação numérica da ortogonalidade

A condição de ortogonalidade (Eq. 4.84 do Beck, com $\beta_n = n\pi/L$) é

$$\int_0^L \sin\!\left(\frac{\beta_n x}{L}\right) \sin\!\left(\frac{\beta_m x}{L}\right) dx = \begin{cases} L/2 & m = n,\\ 0 & m \ne n. \end{cases}$$

Verificamos numericamente construindo a matriz $G_{mn}$ dos produtos internos.

In [ ]:
from scipy.integrate import quad

def inner(m, n, L=L):
    f = lambda x: np.sin(m*np.pi*x/L) * np.sin(n*np.pi*x/L)
    val, _ = quad(f, 0.0, L)
    return val

G = np.array([[inner(m, n) for n in range(1, M+1)] for m in range(1, M+1)])

df_ortog = pd.DataFrame(G,
                        index  =[f'm={i}' for i in range(1, M+1)],
                        columns=[f'n={j}' for j in range(1, M+1)])
pd.set_option('display.float_format', '{:.3e}'.format)
print(df_ortog.to_string())
print(f'\nValor esperado na diagonal: L/2 = {L/2:.3e}')

A diagonal vale $L/2 = 0{,}5$ e os elementos fora da diagonal são da ordem de $10^{-16}$ — ruído de quadratura numérica. A ortogonalidade está numericamente verificada.

## Função de Green $G_{X11}$ e convergência da série

A função de Green transiente é (Eq. 4.X do livro)

$$G_{X11}(x,t \mid x',\tau) = \frac{2}{L} \sum_{n=1}^{\infty} e^{-\beta_n^2 \alpha (t-\tau)/L^2} \sin\!\left(\frac{\beta_n x}{L}\right) \sin\!\left(\frac{\beta_n x'}{L}\right).$$

O **tempo adimensional** $\tilde{\tau} = \alpha (t-\tau)/L^2$ controla a velocidade de convergência: para $\tilde{\tau}$ grande o decaimento exponencial $e^{-n^2\pi^2 \tilde{\tau}}$ trunca a série em poucos modos; para $\tilde{\tau}$ pequeno são necessários muitos modos (este é o regime em que vale a pena usar a forma fechada por imagens — assunto da seção §4.2 do Beck, Tabela 4.1).

In [ ]:
def G_X11(x, xp, tau_tilde, N, L=L):
    """Soma parcial dos N primeiros modos de G_X11 (forma adimensional)."""
    n  = np.arange(1, N+1)
    bn = n * np.pi / L
    expo = np.exp(-(bn*L)**2 * tau_tilde)             # = exp(-n²π² τ̃)
    return (2.0/L) * np.sum(expo * np.sin(bn*x) * np.sin(bn*xp))

# verificação rápida
for tt in [0.5, 0.1, 0.01]:
    val = G_X11(0.5, 0.5, tt, N=200)
    print(f'G_X11(L/2, L/2, τ̃={tt:.3f}, N=200) = {val:.6f}')

### Convergência: erro $|G_N - G_{N_{\max}}|$ em função de $N$

Usamos $N_{\max} = 200$ como **forma fechada da série** (somatório exaustivo, não como aproximação numérica) para medir o quanto cada soma parcial $G_N$ se afasta dela em diferentes regimes de $\tilde{\tau}$.

In [ ]:
Ns        = np.arange(1, 51)
tau_list  = [1.0, 0.1, 0.01, 0.001]
x_eval    = xp_eval = 0.5 * L

fig, ax = plt.subplots()
estilos = ['-', '--', '-.', ':']
for k, tt in enumerate(tau_list):
    G_ref  = G_X11(x_eval, xp_eval, tt, N=200)
    erros  = np.array([abs(G_X11(x_eval, xp_eval, tt, N=N) - G_ref) for N in Ns])
    ax.semilogy(Ns, erros + 1e-18, estilos[k], lw=1.8,
                label=fr'$\tilde{{\tau}} = {tt}$')
ax.set_xlabel(r'$N$ (número de modos)')
ax.set_ylabel(r'$|G_N - G_{200}|$  no centro da placa')
ax.set_title('Convergência da série $G_{X11}$')
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()

## Observações finais

- Para o GX11 **não há equação transcendental a resolver**: os autovalores $\xi_n = n\pi$ saem em forma fechada, exatos em precisão de máquina, sem método numérico (Brent, Newton-Raphson) nem aproximação assintótica (Beck 1992).
- A ortogonalidade $\int_0^L \sin(n\pi x/L)\sin(m\pi x/L)\,dx = (L/2)\,\delta_{nm}$ foi verificada numericamente.
- A convergência da série de $G_{X11}$ é **rápida para $\tilde{\tau} = \alpha(t-\tau)/L^2 \gtrsim 0{,}1$** (poucos modos bastam), mas **lenta para $\tilde{\tau} \to 0$** (regime de tempos curtos), onde é preferível usar a forma fechada equivalente obtida pelo método das imagens (Tabela 4.1 do Beck), que será o assunto do *notebook* `gx11_formas_fechadas.ipynb`.